## Super events and batching candidates

**Information Need:** Identify exact temporal coincidence patterns in an event log, including sets of different activity types occurring simultaneously within the same case (super events) and occurrences of the same activity type occurring simultaneously across multiple cases (batch events).

**Motivation:** Events sharing an exact timestamp can reveal different structural characteristics of an event log. Within a case, multiple activity types recorded at the same moment may represent components of one jointly recorded occurrence and may therefore be candidates for representation as a super event. Across cases, repeated occurrences of the same activity type at one moment may indicate batch processing, scheduled execution, or a shared system operation.

**Precondition:** A minimum activity-set size for super-event detection and a minimum number of distinct cases for batch-event detection are specified.

**Approach:** Analyze timestamp coincidence from two perspectives. For super events, group events by case and exact timestamp, derive the set of distinct activity types in each timestamp window, and retain sets meeting the minimum size. Aggregate identical sets and calculate their timestamp-level occurrences, number of cases, case support, conditional set coverage, and average event coverage. For batch events, group events by activity type and exact timestamp across the complete log, count the distinct participating cases, retain groups meeting the minimum case threshold, and record their participating case identifiers. Rank super-event candidates by their support and coverage measures and batch events by the number of participating cases.

**Output:** Two complementary results: (1) a ranked collection of candidate super events and (2) a ranked collection of candidate batch events.

In [ ]:
import pandas as pd
import pm4py

# --- Configuration -----------------------------------------------------------

LOG_PATH = "../../data/Road_Traffic_Fine_Management_Process.xes"

CASE_ID = 'case:concept:name'
ACTIVITY = 'concept:name'
TIMESTAMP = 'time:timestamp'


MIN_SET_SIZE = 2  # Minimum set size for super-event detection
MIN_BATCH_CASES = 30  # Minimum distinct cases for batch-event detection


In [ ]:
event_log = pm4py.read_xes(LOG_PATH)


#ensure timestamp conversion (Timestamp Normalization pattern)
event_log[TIMESTAMP] = pd.to_datetime(event_log[TIMESTAMP], utc=True, errors='coerce')
#ensure ordering (Event-ordering case-wise)
event_log = event_log.sort_values([CASE_ID, TIMESTAMP]).reset_index(drop=True)

print(f"Cases: {event_log[CASE_ID].nunique():,} | Activities: {event_log[ACTIVITY].nunique():,}")
display(event_log.head())

### 1. Super Events

*The simultaneous occurrence of events of distinct activity types within a case, if sufficiently supported by observations across multiple cases, may indicate the existence of an (implicit) super event.*

In [ ]:
##For each case and timestamp, find the set of events that happened together. Sets meeting the configured minimum size are treated as candidate super events.

#Build timestamp-level event sets
window_df = (
    event_log.groupby([CASE_ID, TIMESTAMP])[ACTIVITY]
    .agg(lambda s: frozenset(sorted(set(s))))
    .reset_index(name='event_set')
)
window_df = window_df[window_df['event_set'].map(len) >= MIN_SET_SIZE].copy()
window_df['set_size'] = window_df['event_set'].map(len)

print(f'Timestamp windows with set size >= {MIN_SET_SIZE}: {len(window_df):,}')
display(window_df.head())

In [ ]:
#Aggregate concurrent sets
set_summary = (  # Aggregate each concurrent event set.
    window_df.groupby('event_set')  # Group identical concurrent sets.
    .agg(occurrences=(CASE_ID, 'size'), cases_with_set=(CASE_ID, 'nunique'))  # Count windows and cases.
    .reset_index()  # Move grouped keys back to columns.
)
set_summary

In [ ]:
#Compute totals
total_cases = event_log[CASE_ID].nunique()  # Count all cases in the log.
event_totals = event_log[ACTIVITY].value_counts().to_dict()  # Count total occurrences per activity.

print(f'Total cases: {total_cases:,}')
print(f'occurrences: {event_totals}')

In [ ]:
#Build case-level activity universes
case_activity_sets = event_log.groupby(CASE_ID)[ACTIVITY].agg(lambda s: set(s)).tolist()  # Build each case's activity universe.

#Enrich set descriptors
set_summary['set_size'] = set_summary['event_set'].map(len)  # Compute set size.

#Case support
# Compute Case support: how common the concurrent set is across all cases.
set_summary['case_support'] = set_summary['cases_with_set'] / total_cases 

#Set coverage
#Compute Set coverage: among cases where all events in the set appear, how often they appear concurrently.

set_summary['cases_with_all_events'] = set_summary['event_set'].map(  # Count cases where all set events appear.
    lambda s: sum(s.issubset(case_set) for case_set in case_activity_sets)
 )
set_summary['set_coverage'] = (set_summary['cases_with_set'] / set_summary['cases_with_all_events']).fillna(0.0) 

In [ ]:
#Event coverage
# Compute Event coverage: for each set, on average, how frequent its events are relative to their individual total occurrences in the log.
if set_summary.empty:
    set_summary['event_coverage_avg'] = pd.Series(
        index=set_summary.index,
        dtype='float64',
    )
else:
    set_summary['event_coverage_avg'] = set_summary.apply(
        lambda r: (
            sum(
                r['occurrences'] / event_totals[e]  # Event-level share covered by this concurrent set.
                for e in r['event_set']  # Iterate through events in the current set.
            )
            / len(r['event_set'])  # Average over set size (number of events in the set), otherwise the coverage would be biased towards larger sets (higher sums with set having more elements)
        ),
        axis=1,  # Apply row-wise: one concurrent set at a time.
    )

In [ ]:
#Final ranking output
set_summary = (  # Select and order final result columns.
    set_summary[['event_set', 'case_support', 'set_coverage', 'event_coverage_avg', 'occurrences']]
    .sort_values(['case_support', 'set_coverage', 'event_coverage_avg', 'occurrences'], ascending=False)
    .reset_index(drop=True)
)
print(f'Candidate super events: {len(set_summary):,}')  # Report result size.
set_summary

### 2. Cross-case batch events

*A batch event is an activity recorded at the same timestamp in multiple distinct cases.*

In [ ]:
#Detect cross-case batch events
batch_events = (
    event_log.groupby([ACTIVITY, TIMESTAMP])
    .agg(
        case_count=(CASE_ID, 'nunique'),
        case_ids=(CASE_ID, lambda s: tuple(sorted(s.astype(str).unique()))),
    )
    .reset_index()
    .query('case_count >= @MIN_BATCH_CASES')
    .sort_values(['case_count', ACTIVITY, TIMESTAMP], ascending=[False, True, True])
    .reset_index(drop=True)
)

print(f'Batch events spanning at least {MIN_BATCH_CASES} cases: {len(batch_events):,}')
batch_events